In [ ]:
# Installe d'abord dans une cellule : !pip install -q transformers peft bitsandbytes accelerate datasets scikit-learn

import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

MODEL_ID = "Qwen/Qwen2.5-3B" # Léger et très performant sur GPU T4 gratuit

df = pd.read_csv("darija_clean.csv")
labels_list = sorted(df['label'].unique())
label2id = {l: i for i, l in enumerate(labels_list)}
df['label_id'] = df['label'].map(label2id)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(labels_list), quantization_config=bnb_config, device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="SEQ_CLS"
)
model = get_peft_model(model, lora_config)

def tokenize_fn(batch):
    return tokenizer(batch["clean_text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'})).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'})).map(tokenize_fn, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./results_qwen",
        eval_strategy="epoch",
        per_device_train_batch_size=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True
    ),
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()
print("Résultats Qwen-2.5 QLoRA :", trainer.evaluate())